# Steer — KSteer / CAA / LoReFT activation steeringReproduces the **Steering** workflow from §3.2 of the DreamReader paper(Fig. 2 / Fig. 3 case studies). Trains a steering vector / adapter onpaired activations, then injects it during generation.Config comes from `t2i_interp/config/steer/{run,caa,ksteer,loreft}.yaml`.Equivalent CLI: `t2i-steer --config-name=steer/loreft model=sdxl_turbo`.

In [ ]:
# Bootstrap: locate repo root, put on sys.path, chdir there.
# Works whether you launched Jupyter from repo root or from notebooks/.
from pathlib import Path
import os, sys

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Could not locate repo root (no pyproject.toml found up the tree).")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print(f"Repo root: {ROOT}")

In [ ]:
# Hydra compose: model preset + auto-detected device/dtype.
# Change MODEL_PRESET below to "sd15" / "sdxl" / "sdxl_turbo".
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf
from t2i_interp.config._hydra_config import config_dir
import torch

MODEL_PRESET = "sd15"  # ← change me

if torch.cuda.is_available():
    DEVICE, DTYPE = "cuda:0", "float16"
elif torch.backends.mps.is_available():
    DEVICE, DTYPE = "mps", "bfloat16"
else:
    DEVICE, DTYPE = "cpu", "float32"

with initialize_config_dir(config_dir=config_dir(), version_base=None):
    cfg = compose(
        config_name="steer/run",
        overrides=[
            f"model={MODEL_PRESET}",
            f"device={DEVICE}",
            f"dtype={DTYPE}",
            "wandb.project=null",
        ],
    )
print(OmegaConf.to_yaml(cfg))

In [ ]:
import os
import torch
from datasets import load_dataset
from diffusers import AutoPipelineForText2Image

from t2i_interp.linear_steering import KSteer
from t2i_interp.mapper import MLPMapperTwoHeads
from t2i_interp.t2i import T2IModel
from t2i_interp.utils.T2I.buffer import ActivationsDataloader, PairedLoader
from t2i_interp.utils.T2I.collect_latents import collect_latents

# Override per-notebook research vars here (or set on the CLI via overrides=)
dataset_name = "nirmalendu01/socialcounterfactuals-1000"
layer_name = "unet.up_blocks.2.attentions.1.transformer_blocks.0.attn2"
prompt_col = "caption"
race_col = "race"
gender_col = "gender"
save_dir = "./latents_cache/social_steering"

model = T2IModel(
    cfg.model_key,
    automodal=AutoPipelineForText2Image,
    device=cfg.device,
    dtype=cfg.dtype,
)

In [ ]:
import os

os.chdir("..")  # should be run from repo root
import torch
from datasets import load_dataset
from diffusers import AutoPipelineForText2Image

from t2i_interp.linear_steering import KSteer
from t2i_interp.mapper import MLPMapperTwoHeads
from t2i_interp.t2i import T2IModel
from t2i_interp.utils.T2I.buffer import ActivationsDataloader, PairedLoader
from t2i_interp.utils.T2I.collect_latents import collect_latents

# Config
dataset_name = "nirmalendu01/socialcounterfactuals-1000"
model_key = "runwayml/stable-diffusion-v1-5"
layer_name = "unet.up_blocks.2.attentions.1.transformer_blocks.0.attn2"
prompt_col = "caption"
race_col = "race"
gender_col = "gender"
save_dir = "./latents_cache/social_steering"
device = "cuda" if torch.cuda.is_available() else "cpu"
init_seed = 42
# Initialize model once
model = T2IModel(model_key, automodal=AutoPipelineForText2Image, device=device, dtype="float16")

## 1. Prepare Dataset and Collect Latents

In [ ]:
print("Loading dataset...")
ds_full = load_dataset(dataset_name)
ds_train = ds_full["train"]
ds_val = (
    ds_full["validation"]
    if "validation" in ds_full
    else ds_full["test"]
    if "test" in ds_full
    else None
)

# If no val split, create one from train (80/20)
if ds_val is None:
    split = ds_train.train_test_split(test_size=0.2, seed=42)
    ds_train = split["train"]
    ds_val = split["test"]

# Create labels from train (use same mapping for val)
races = sorted(list(set(ds_train[race_col])))
genders = sorted(list(set(ds_train[gender_col])))
race2idx = {r: i for i, r in enumerate(races)}
gender2idx = {g: i for i, g in enumerate(genders)}

print(f"Races ({len(races)}): {race2idx}")
print(f"Genders ({len(genders)}): {gender2idx}")
print(f"Train: {len(ds_train)}, Val: {len(ds_val)}")


def add_labels(example):
    example["race_label"] = race2idx[example[race_col]]
    example["gender_label"] = gender2idx[example[gender_col]]
    return example


ds_train = ds_train.map(add_labels)
ds_val = ds_val.map(add_labels)

In [ ]:
# Collect Latents - Train
print("Collecting train latents...")
train_save_path = collect_latents(
    accessors=[layer_name],
    dataset=ds_train,
    model=model,
    save_path=os.path.join(save_dir, "train"),
    columns=[prompt_col],
    batch_size=4,
    guidance_scale=7.5,
    conditional_only=True,
)

# Collect Latents - Val
print("Collecting val latents...")
val_save_path = collect_latents(
    accessors=[layer_name],
    dataset=ds_val,
    model=model,
    save_path=os.path.join(save_dir, "val"),
    columns=[prompt_col],
    batch_size=4,
    guidance_scale=7.5,
    conditional_only=True,
)

## 2. Train Mapper

In [ ]:
# Create Loaders
print("Creating loaders...")
extra_keys = ["race_label.pth", "gender_label.pth"]

# Check if paths are defined, else find latest
if "train_save_path" not in locals():
    train_dir = os.path.join(save_dir, "train")
    # Sort by name (timestamp)
    if os.path.exists(train_dir):
        subdirs = sorted(
            [d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))]
        )
        if subdirs:
            train_save_path = os.path.join(train_dir, subdirs[-1])
            print(f"Using executed train path: {train_save_path}")
        else:
            raise ValueError(f"No train latents found in {train_dir}")
    else:
        raise ValueError(f"Train dir {train_dir} does not exist")

if "val_save_path" not in locals():
    val_dir = os.path.join(save_dir, "val")
    if os.path.exists(val_dir):
        subdirs = sorted(
            [d for d in os.listdir(val_dir) if os.path.isdir(os.path.join(val_dir, d))]
        )
        if subdirs:
            val_save_path = os.path.join(val_dir, subdirs[-1])
            print(f"Using executed val path: {val_save_path}")
        else:
            # Fallback to train path if val missing? Or just error.
            print(f"No val latents found in {val_dir}, using train path as dummy val")
            val_save_path = train_save_path
    else:
        val_save_path = train_save_path  # Fallback

train_tar_path = os.path.join(train_save_path, f"{layer_name}_{prompt_col}.tar")
val_tar_path = os.path.join(val_save_path, f"{layer_name}_{prompt_col}.tar")

train_act_loader = ActivationsDataloader(
    paths_to_datasets=[train_tar_path], block_name=layer_name, batch_size=16, device=device
)

val_act_loader = ActivationsDataloader(
    paths_to_datasets=[val_tar_path], block_name=layer_name, batch_size=16, device=device
)

train_race_loader = ActivationsDataloader(
    [train_tar_path], layer_name, 16, data_key="race_label.pth", flatten=False, seed=init_seed
)
train_gender_loader = ActivationsDataloader(
    [train_tar_path], layer_name, 16, data_key="gender_label.pth", flatten=False, seed=init_seed
)

train_loader = PairedLoader([train_act_loader, train_race_loader, train_gender_loader])

val_race_loader = ActivationsDataloader(
    [val_tar_path], layer_name, 16, data_key="race_label.pth", flatten=False, seed=init_seed
)
val_gender_loader = ActivationsDataloader(
    [val_tar_path], layer_name, 16, data_key="gender_label.pth", flatten=False, seed=init_seed
)
val_loader = PairedLoader([val_act_loader, val_race_loader, val_gender_loader])

In [ ]:
# Fit
print("Fitting...")
ksteer = KSteer(model=model)

# SD1.5 down_blocks.2 -> 640 dim
input_dim = 640
steer_mapper = MLPMapperTwoHeads(input_dim=input_dim, output_dims=[len(races), len(genders)])

from t2i_interp.utils.training import Training, TrainingSpec

spec = TrainingSpec(
    training_function=ksteer.fit,
    kwargs={
        "train_loader": train_loader,
        "val_loader": val_loader,
        "mapper": steer_mapper,
        "loss_fn": torch.nn.CrossEntropyLoss(),
        "train_steps": 100,
        "lr": 1e-4,
    },
)
trainer = Training(spec)
output = trainer.run_trainer()

## 3. Generate Steered Images

In [ ]:
import os

from t2i_interp.utils.inference import Inference, InferenceSpec
from t2i_interp.utils.plot import show_grid  # <--- Import from utils

# --- Setup Output ---
output_dir = "output_images"
os.makedirs(output_dir, exist_ok=True)
print(f"Saving images to: {output_dir}")


prompts = ["A photo a doctor", "A portrait of a person"]


def run_steer(target_idx, avoid_idx, alpha, steer_steps, layer_name, desc):
    print(f"  -> {desc}")
    all_images = []
    all_labels = []
    # try:
    imgs = ksteer.steer(
        prompts,
        target_idx=target_idx,
        avoid_idx=avoid_idx,
        alpha=alpha,
        layer_name=layer_name,
        steer_steps=steer_steps,
    )
    all_images.extend(imgs)
    all_labels.extend([f"{p}\n({desc})" for p in prompts])
    return all_images, all_labels


specs = []
alpha = 10
steer_steps = 5

specs.extend(
    [
        InferenceSpec(
            name="towards_asian",
            inference_fn=run_steer,
            kwargs={
                "target_idx": [[race2idx["Asian"]], None],
                "avoid_idx": None,
                "alpha": alpha,
                "steer_steps": steer_steps,
                "layer_name": layer_name,
                "desc": "Towards Asian",
            },
        ),
        InferenceSpec(
            name="towards_female",
            inference_fn=run_steer,
            kwargs={
                "target_idx": [None, [gender2idx["female"]]],
                "avoid_idx": None,
                "alpha": alpha,
                "steer_steps": steer_steps,
                "layer_name": layer_name,
                "desc": "Towards Female",
            },
        ),
        InferenceSpec(
            name="towards_asian_female",
            inference_fn=run_steer,
            kwargs={
                "target_idx": [[race2idx["Asian"]], [gender2idx["female"]]],
                "avoid_idx": None,
                "alpha": alpha,
                "steer_steps": steer_steps,
                "layer_name": layer_name,
                "desc": "Towards Asian+Female",
            },
        ),
        InferenceSpec(
            name="away_male",
            inference_fn=run_steer,
            kwargs={
                "target_idx": None,
                "avoid_idx": [None, [gender2idx["male"]]],
                "alpha": alpha,
                "steer_steps": steer_steps,
                "layer_name": layer_name,
                "desc": "Away from Male",
            },
        ),
        InferenceSpec(
            name="away_white",
            inference_fn=run_steer,
            kwargs={
                "target_idx": None,
                "avoid_idx": [[race2idx["White"]], None],
                "alpha": alpha,
                "steer_steps": steer_steps,
                "layer_name": layer_name,
                "desc": "Away from White",
            },
        ),
        InferenceSpec(
            name="away_white_male",
            inference_fn=run_steer,
            kwargs={
                "target_idx": None,
                "avoid_idx": [[race2idx["White"]], [gender2idx["male"]]],
                "alpha": alpha,
                "steer_steps": steer_steps,
                "layer_name": layer_name,
                "desc": "Away from White+Male",
            },
        ),
    ]
)

out = []
for spec in specs:
    inference = Inference(spec)
    out.append(inference.run_inference())

# --- Plot All ---
show_grid(
    [img for item in out for img in item.preds[0]],
    [lbl for item in out for lbl in item.preds[1]],
    cols=3,
)

# CAA Steer

In [ ]:
import torch as th

from t2i_interp.linear_steering import CAA
from t2i_interp.utils.training import Training, TrainingSpec


# --- 1. Helper to Collect Activations from Loaders ---
def collect_from_loader(loader, target_labels=None, max_samples=1000, desc="Collecting"):
    """
    Iterates through the loader and collects activations matching target_labels.
    If target_labels is None, collects all.
    """
    collected = []
    count = 0

    # Ensure target_labels is a list if provided
    if target_labels is not None and not isinstance(target_labels, list):
        target_labels = [target_labels]

    print(desc)

    # Iterate through loader
    iterator = loader.iterate()

    try:
        while count < max_samples:
            batch_data = next(iterator)

            # Unpack batch (Adapter for Paired/Activations loader)
            if isinstance(batch_data, (tuple, list)):
                act = batch_data[0]
                label = batch_data[1] if len(batch_data) > 1 else None
            else:
                act = batch_data
                label = None

            B = act.shape[0]

            # Handle label format (might be list or tensor)
            if isinstance(label, list):
                label = label[0]  # Handle extra_keys list wrapper
            if th.is_tensor(label):
                label = label.cpu().tolist()
            if not isinstance(label, list) and label is not None:
                label = [label] * B

            # Filter and collect
            for i in range(B):
                l = label[i] if label else None
                if th.is_tensor(l):
                    l = l.item()

                if isinstance(l, list):
                    l = l[0]
                if target_labels is None or l in target_labels:
                    collected.append(act[i].detach().cpu())
                    count += 1

                if count >= max_samples:
                    break
    except StopIteration:
        pass

    print(f"  Done. Got {len(collected)} samples.")

    if not collected:
        return th.tensor([])  # Fail safe

    return th.stack(collected)


# --- 2. Fit CAA Vectors ---
print("\n--- Fitting CAA ---")
caa = CAA(model=model)


train_loader.reset()
# A. Asian Vector
print("\nFit Asian Vector:")
# Pos: Asian
pos_asian = collect_from_loader(train_loader, [race2idx["Asian"]], desc="  Pos (Race: Asian)")
# Neg: White, Black, Indian (All except Asian) - we explicitly target "Non Asian"
neg_asian = collect_from_loader(
    train_loader,
    [race2idx["White"], race2idx["Black"], race2idx["Indian"]],
    desc="  Neg (Race: Others)",
)


spec = TrainingSpec(
    training_function=caa.fit,
    kwargs={"pos_acts": pos_asian, "neg_acts": neg_asian, "attr_name": "asian_vector"},
)
trainer = Training(spec)
output = trainer.run_trainer()

In [ ]:
from t2i_interp.utils.inference import Inference, InferenceSpec
from t2i_interp.utils.plot import show_grid

prompts = ["A photo a doctor", "A portrait of a person"]


def run_steer(steering_vecs, desc):
    print(f"  -> {desc}")
    try:
        # all_images=[]
        all_labels = []
        imgs = caa.steer(prompts, steering_vecs=steering_vecs, layer_name=layer_name, alphas=[10.0])
        all_labels.extend([f"{p}\n({desc})" for p in prompts])
        return imgs, all_labels
    except Exception as e:
        print(f"Error generating {desc}: {e}")


specs = []
specs.extend(
    [
        InferenceSpec(
            name="towards_asian",
            inference_fn=run_steer,
            kwargs={
                "steering_vecs": caa.steering_vecs["asian_vector"].squeeze(),
                "desc": "Towards Asian",
            },
        ),
    ]
)

out = []
for spec in specs:
    inference = Inference(spec)
    out.append(inference.run_inference())

# --- Plot All ---
show_grid(
    [img for item in out for img in item.preds[0]],
    [lbl for item in out for lbl in item.preds[1]],
    cols=3,
)

## LoREFT Steering

In [ ]:
### create a new dataset from Intel/SocialCounterfactuals (we want to steer generation to black+female)

import regex as re
from datasets import load_dataset

ATTR_COLS = ["a1", "a2"]
FILTER_a1 = "Black"
FILTER_a2 = "female"


def map_caption(example):
    caption = example["caption"]
    caption = caption.replace("A photo of a ", "a headhot of a ")
    example["caption"] = caption
    return example


def build_base_caption(caption: str, attrs: list[str]):
    """
    Remove each attr (as a word/phrase) from caption.
    e.g. "a photo of a skinny male academic"
         attrs = ["skinny", "male"]
         -> "a photo of a academic"
    """
    new_cap = caption
    for attr in attrs:
        if not attr:
            continue
        attr = attr.strip()
        if not attr:
            continue
        # \b attr \b with case-insensitive match
        pattern = r"\b" + re.escape(attr) + r"\b"
        new_cap = re.sub(pattern, " ", new_cap, flags=re.IGNORECASE)

    # collapse extra whitespace
    new_cap = re.sub(r"\s+", " ", new_cap).strip()
    return new_cap


def add_caption_base(example):
    caption = example["caption"]

    attrs = []
    for col in ATTR_COLS:
        if col in example and isinstance(example[col], str):
            a = example[col].strip()
            if a and a.lower() != "none":
                attrs.append(a)

    base = build_base_caption(caption, attrs)
    example["caption_base"] = base
    return example


# 1. Load HF dataset and add caption_base
print("Loading dataset Intel/SocialCounterfactuals...")
ds = load_dataset("Intel/SocialCounterfactuals", split="train")

# filter to only examples with desired attrs
ds = ds.filter(
    lambda ex: (FILTER_a1.lower() in ex["a1"].lower()) and (FILTER_a2.lower() in ex["a2"].lower())
)
ds = ds.map(map_caption)
# add caption_base (remove attrs)
ds = ds.map(add_caption_base)

# optional: filter out examples where caption == caption_base (no attrs)
ds = ds.filter(lambda ex: ex["caption"] != ex["caption_base"])
# rename caption base and caption columns
ds = ds.rename_column("caption_base", "student")
ds = ds.rename_column("caption", "teacher")

In [ ]:
import os

from t2i_interp.linear_steering import LoREEFT
from t2i_interp.utils.T2I.buffer import ActivationsDataloader, PairedLoader
from t2i_interp.utils.T2I.collect_latents import collect_latents
from t2i_interp.utils.training import Training, TrainingSpec

# ── 1. Train / val split ────────────────────────────────────────────
val_frac = 0.1
n_total = len(ds)
n_val = max(1, int(n_total * val_frac))
ds_train = ds.select(range(n_total - n_val))
ds_val = ds.select(range(n_total - n_val, n_total))

print(f"Train: {len(ds_train)} samples  |  Val: {len(ds_val)} samples")

# ── 2. Collect Latents for LoREFT ─────────────────────
layer_idx = 5
capture_layer_name = f"text_encoder.text_model.encoder.layers.{layer_idx}"

save_dir_loreft = "./latents_cache/loreft/clip"
os.makedirs(save_dir_loreft, exist_ok=True)

print("Collecting LoReFT train latents...")
train_save_path_loreft = collect_latents(
    accessors=[capture_layer_name],
    dataset=ds_train,
    model=model,
    save_path=os.path.join(save_dir_loreft, "train"),
    columns=["student", "teacher"],  # "student" was "caption_base" which serves as the base
    batch_size=16,
    guidance_scale=1.0,
    conditional_only=False,
)

print("Collecting LoReFT val latents...")
val_save_path_loreft = collect_latents(
    accessors=[capture_layer_name],
    dataset=ds_val,
    model=model,
    save_path=os.path.join(save_dir_loreft, "val"),
    columns=["student", "teacher"],
    batch_size=16,
    guidance_scale=1.0,
    conditional_only=False,
)

# Train Loaders
train_base_tar = os.path.join(train_save_path_loreft, f"{capture_layer_name}_student.tar")
train_teacher_tar = os.path.join(train_save_path_loreft, f"{capture_layer_name}_teacher.tar")

train_base_loader = ActivationsDataloader([train_base_tar], capture_layer_name, 16, device="cuda:0")
train_teacher_loader = ActivationsDataloader(
    [train_teacher_tar], capture_layer_name, 16, device="cuda:0"
)
train_loader = PairedLoader([train_base_loader, train_teacher_loader])

# Val Loaders
val_base_tar = os.path.join(val_save_path_loreft, f"{capture_layer_name}_student.tar")
val_teacher_tar = os.path.join(val_save_path_loreft, f"{capture_layer_name}_teacher.tar")

val_base_loader = ActivationsDataloader([val_base_tar], capture_layer_name, 16, device="cuda:0")
val_teacher_loader = ActivationsDataloader(
    [val_teacher_tar], capture_layer_name, 16, device="cuda:0"
)
val_loader = PairedLoader([val_base_loader, val_teacher_loader])

# ── 3. Fit LoREEFT via TrainingSpec ─────────────────────────────────
steer = LoREEFT(model)

spec = TrainingSpec(
    training_function=steer.fit,
    kwargs={
        "loader": train_loader,
        "layer_name": "clip",  # uses train_loreft_on_clip
        "val_loader": val_loader,
        "layer_idx": layer_idx,  # CLIP hidden-state layer index
        "rank": 16,
        "num_steps": 1_000,
        "lr": 1e-3,
        "device": "cuda:0",
        "log_steps": 100,
    },
)
trainer = Training(spec)
output = trainer.run_trainer()

In [ ]:
# ── LoREEFT Steering Inference ──────────────────────────────────────
from t2i_interp.utils.inference import Inference, InferenceSpec
from t2i_interp.utils.plot import show_grid

prompts = [
    "A photo of a person",
    "A portrait of a white man",
]

# layer_name must match the layer_idx used during fit (layer_idx=5)
layer_name = "unet.up_blocks.2.attentions.1.transformer_blocks.0.attn2"


def run_loreft_steer(layer_name, num_inference_steps=30, desc="LoREEFT"):
    """Run LoREEFT steer and return (images, labels)."""
    print(f"  -> {desc}")
    imgs = steer.steer(
        prompts=prompts,
        layer_name=layer_name,
        num_inference_steps=num_inference_steps,
    )
    labels = [f"{p}\n({desc})" for p in prompts]
    return imgs, labels


def run_baseline(num_inference_steps=30, desc="baseline"):
    """Run the pipeline without steering."""
    print(f"  -> {desc}")
    imgs = model.pipeline(prompts, num_inference_steps=num_inference_steps).images
    labels = [f"{p}\n({desc})" for p in prompts]
    return imgs, labels


specs = [
    InferenceSpec(
        name="baseline",
        inference_fn=run_baseline,
        kwargs={"num_inference_steps": 30, "desc": "baseline"},
    ),
    InferenceSpec(
        name="loreft_steered",
        inference_fn=run_loreft_steer,
        kwargs={"layer_name": layer_name, "num_inference_steps": 30, "desc": "LoREEFT steered"},
    ),
]

out = []
for spec in specs:
    inference = Inference(spec)
    out.append(inference.run_inference())

# Plot baseline and steered side-by-side
all_imgs = [img for item in out for img in item.preds[0]]
all_labels = [lbl for item in out for lbl in item.preds[1]]
show_grid(all_imgs, all_labels, cols=len(prompts))

In [ ]:
# Run fingerprint: writes a reproducibility record (model, dataset, intervention,
# git SHA, full config) to ./notebook_runs/steer/fingerprint.json.
# Same machine-independent hash as the CLI workflow.
from t2i_interp.reporting.fingerprint import RunFingerprint

fp = RunFingerprint.from_cfg(
    cfg,
    workflow="steer",
    intervention={
        "steer_type": cfg.get("steer_type"),
        "layer_name": "unet.up_blocks.2.attentions.1.transformer_blocks.0.attn2",
        "alpha": cfg.get("alpha"),
    },
)
fp.write(f"./notebook_runs/steer/fingerprint.json")
print(f"Fingerprint: {fp.hash()}  ({fp.workflow})")